**Timeline**

* ТГ-бот. UI: меню, кнопки, связь с разработчиками
* FastAPI для мониторинга



In [ ]:
!apt-get install tesseract-ocr -y
!apt-get install tesseract-ocr-rus -y
!pip install pytesseract pandas python-telegram-bot==20.7 opencv-python-headless psycopg2-binary fuzzywuzzy python-Levenshtein
!pip install openai --upgrade -q

print("Библиотеки установлены")

In [ ]:
import os, re, asyncio
import numpy as np
import io, logging, random
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes
import nest_asyncio
from PIL import Image, ImageEnhance
import pytesseract, cv2, psycopg2
from psycopg2 import pool
from fuzzywuzzy import fuzz, process
from openai import OpenAI
from google.colab import userdata

nest_asyncio.apply()
logging.basicConfig(level=logging.ERROR)

pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'
BOT_TKN = userdata.get('BOT_TOKEN') # лежит в секретах коллаба

OPENAI_KEY = userdata.get('OPENAI_API_KEY') # в секретах
client = OpenAI(api_key=OPENAI_KEY)

checks = 0

print("Всё готово")

In [ ]:
DB_HOST = "eco-scan-ecobot.e.aivencloud.com" # бд хранится в aiven
DB_PORT = "10408"
DB_NAME = "defaultdb"
DB_USER = "avnadmin"
DB_PASS = userdata.get('DB_PASSWORD') # в секретах

db_Pool = psycopg2.pool.SimpleConnectionPool(1, 10, host=DB_HOST, port=DB_PORT, database=DB_NAME, user=DB_USER, password=DB_PASS, sslmode='require')
print("Подключено к БД")

def load():
    conn = db_Pool.getconn()
    try:
        with conn.cursor() as cursor:
            cursor.execute("""SELECT name, matched_product_id FROM lenta WHERE name IS NOT NULL
                UNION ALL SELECT name, matched_product_id FROM magnit WHERE name IS NOT NULL
                UNION ALL SELECT name, matched_product_id FROM okey WHERE name IS NOT NULL
                UNION ALL SELECT name, matched_product_id FROM perekrestok WHERE name IS NOT NULL
                UNION ALL SELECT name, matched_product_id FROM x5 WHERE name IS NOT NULL""")
            store_prods = {}
            for name, matched_id in cursor.fetchall():
                if name and len(name) > 3:
                    store_prods[name.lower()] = matched_id

            cursor.execute("""SELECT pr.id, pr.name, c.name as category, pr.carbon_footprint FROM product_reference pr JOIN categories c ON pr.category_id = c.id""")
            products = {}
            for product_id, name, category, footprint in cursor.fetchall():
                products[product_id] = {'name': name, 'category': category, 'co2': float(footprint) if footprint else 0.0}
            print(f"Загружено названий из магазинов: {len(store_prods)}")
            print(f"Загружено продуктов из справочника: {len(products)}")
            return store_prods, products
    except:
        print("Ошибка")
        return {}, {}
    finally:
        db_Pool.putconn(conn)

store_prods, prod_info = load()

In [ ]:
def preproc_img(image_bytes):
    print('предобработка')
    try:
        nparr = np.frombuffer(image_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        if img == None:
            return None

        height, width = img.shape[:2]
        if width < 1500:
            scale = 2000 / width
            new_width = int(width * scale)
            new_height = int(height * scale)
            img = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_CUBIC)

 # оттенки серого
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))      # чтобы лучше обрабатывать фотки с плохим освещением
        enhanced = clahe.apply(gray)
# очистка от шума
        denoised = cv2.fastNlMeansDenoising(enhanced, h=30)
# перевод в чб
        binary = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

        pil_img = Image.fromarray(binary)
# резкость
        sharpen = ImageEnhance.Sharpness(pil_img)
        shrpnd = sharpen.enhance(1.5)
        return shrpnd
    except:
        print("Ошибка предобработки")
        return None

In [ ]:
# промпт для гпт

async def gpt(ocr_txt):
    print("Отправка в GPT")
    if not OPENAI_KEY:
        print("Нет ключа")
        return ocr_txt

    try:
        prompt = f"""Ты эксперт по исправлению OCR-ошибок в чеках. ВАЖНЫЕ ПРАВИЛА:
1. Текст чеков на РУССКОМ языке. Не переводи ничего на английский или другие языки.
2. ТОРГОВЫЕ НАИМЕНОВАНИЯ (бренды, названия продуктов) НЕЛЬЗЯ заменять или переводить.
3. Исправляй только явные опечатки (буквы, цифры).
4. НЕ удаляй пробелы и дефисы в названиях.
5. Сохраняй оригинальную структуру строк.
6. Не переводи и не переименовывай продукты.
7. Сохраняй КАЖДУЮ строку чека.
8. НЕ УДАЛЯЙ и НЕ ОБЪЕДИНЯЙ строки с продуктами.

Оригинальный OCR-текст:
{ocr_txt}

Верни ТОЛЬКО исправленный текст, сохраняя все строки, без пояснений и комментариев."""

        response = client.chat.completions.create(model="gpt-4.1-mini", messages=[{"role": "user", "content": prompt}], max_tokens=2000, temperature=0.1)

        fxd_txt = response.choices[0].message.content
        return fxd_txt
    except:
        print("Ошибка")
        return ocr_txt

In [ ]:
async def img_txt(photo_file):
    try:
        file = await photo_file.get_file()
        ph_bytes = await file.download_as_bytearray()
        processed = preproc_img(ph_bytes)
        if processed:
# три режима для лучшего распознавания
            text1 = pytesseract.image_to_string(processed, lang='rus+eng', config='--oem 3 --psm 6')
            text2 = pytesseract.image_to_string(processed, lang='rus+eng', config='--oem 3 --psm 3')
            text3 = pytesseract.image_to_string(processed, lang='rus+eng', config='--oem 3 --psm 11')
            combined = text1 + "\n" + text2 + "\n" + text3

            combined = await gpt(combined)

            return combined.lower()
        return ""
    except:
        print("Ошибка")
        return ""

In [ ]:
PROBLEMS = {'gal.пастила h.s.p.мед.обл.': ['салатила', 'салат пастила', 'пастила н.', 's.p.мед', 's.p.нед', 'н.5.р.нец', 'h.s.p.мед'],
    'nina farina тараллини с чесноком': ['tanai', 'farina', 'sf j)nina', 'taran', 'taranfvh'],
    'святой источник вода питьевая лимон': ['свят ион', 'свято', 'святой'],
    'хлебцы молод.лайт с вино.кос': ['executor', 'honoo', 'aaat', 'кгс7оф'],
    'салат фрилис': ['фрилис', 'phankc', 'opnankc', 'фрилик', 'фрилиc', 'салат фрилис'],
    'хлебец зож': ['xnebeu', 'зож', 'body-эф', 'body', 'хлебец зож'],
    'mentos жев.рез.p.fr.з.м.15,5г' : ['mentos', 'heb.pe3.p.fr.3.m.15'],
    'gl.vil.нект.м/фр.из см.фруко': ['фруко', 'см.фруко'],
    'джин gletcher 40% 0.25л' : ['gletcher', "'gletcher"],
    'ml.s.сыр серб.брынза мяг.': ['брынза', 'серб.брынза', 'ml.s.сыр', 'сыр серб', 'брынза мяг', 'аззне'],
    'lor.чипсы nat.мор.сол/перец.' : ['lor.чипсы', 'nat.мор.'],
    'пиво варим сусло св.нефил.' : ['варим сусло', 'baphn cycno', 'cb.heohn'],
    'ром выдержанный steersman' : ['steersman', 'ром'],}

In [ ]:
# ложно-положит срабатывание

def fake(prod_name, receipt_line, match_score=100):
    prod_l = prod_name.lower()
    receipt_l = receipt_line.lower()

    if 'томат' in prod_l or 'помидор' in prod_l:
        if 'томат' not in receipt_l and 'помидор' not in receipt_l and 'черри' not in receipt_l:
            print(f"Отфильтрован 'Томаты' (в строке нет 'томат'/'черри')")
            return True

   # порог подобран методом научного тыка
    if match_score < 55:
        return True

    if 'святой' in prod_l and match_score < 75:
        print(f"Отфильтрован 'Святой источник' (скор: {match_score}%)")
        return True

    return False

In [ ]:
def find_prods(text):
    found = []
    found_prods = set()

    print("\nПоиск продуктов")

    for prod_name, aliases in PROBLEMS.items():
        for alias in aliases:
            if alias in text:
                product_id = store_prods.get(prod_name)
                if product_id and product_id in prod_info:
                    if fake(prod_name, alias, 100):
                        continue
                    product = prod_info[product_id]
                    if prod_name not in found_prods:
                        found.append({'receipt_name': prod_name, 'category': product['category'], 'co2': product['co2']})
                        found_prods.add(prod_name)
                        print(f"Найдено по алиасу: {prod_name[:40]}")
                    break

    lines = text.split('\n')
    candidates = []
    for line in lines:
        line = line.strip()
        if not line or len(line) < 4:
            continue
        clean = re.sub(r'\d+', '', line)
        clean = re.sub(r'[^\w\s]', ' ', clean)
        clean = re.sub(r'\s+', ' ', clean).strip().lower()
        if len(clean) > 3:
            candidates.append(clean)

    all_names = list(store_prods.keys())

    for c in candidates:
        already_f = any(f['receipt_name'] in c for f in found)
        if already_f:
            continue

        if len(c) < 5:
            continue

        if not re.search(r'[а-яё]', c):
            continue

        cand_len = len(c)
        if cand_len < 10:
            dynamic_thr = 80
        elif cand_len > 30:
            dynamic_thr = 55
        else:
            dynamic_thr = 65

        mtchs = process.extract(c, all_names, limit=1, scorer=fuzz.ratio)

        for mtch, score in mtchs:
            if score >= dynamic_thr:
                if fake(mtch, c, score):
                    print(f"Отфильтровао: {mtch} ({score}%)")
                    continue

# глюк бота, находит, даже если нет в чеке

                if 'банан' in mtch.lower():
                    pastila = any('пастила' in f['receipt_name'].lower() for f in found)
                    if pastila or 'салатила' in c or 'листила' in c:
                        print(f"Блокировка: {mtch}")
                        continue

                if 'томат' in mtch.lower():
                    txt_low = text.lower()
                    if 'томат' not in txt_low and 'черри' not in txt_low and 'помидор' not in txt_low:
                        print(f"Блокировка ложных томатов: {mtch} (в тексте нет признаков томатов)")
                        continue

                if 'магнит' in mtch.lower():
                    keywrds = ['сок', 'ролл', 'напиток', 'энергет', 'пиво', 'шоколад', 'сыр', 'пастила', 'хлеб']
                    prod_keyw = any(keyword in c for keyword in keywrds)
                    if not prod_keyw:
                        print(f"Блокировка ложного продукта 'Магнит': {mtch} (нет ключевых слов продукта)")
                        continue

                product_id = store_prods.get(mtch)
                if product_id and product_id in prod_info:
                    if mtch in found_prods:
                        print(f"Пропуск точного дубля: {mtch}")
                        continue
                    product = prod_info[product_id]
                    found.append({'receipt_name': mtch, 'category': product['category'], 'co2': product['co2']})
                    found_prods.add(mtch)
                    print(f"Найдено ({score}%): {mtch[:40]}")
                    break

    return found

In [ ]:
hightips = ["🛍️ Старайтесь выбирать товары в бумажной упаковке вместо пластиковой — это снижает углеродный след.",
    "🥕 Добавьте в рацион больше растительных продуктов — их производство экологичнее.",
    "🚫 Откажитесь от товаров с избыточной упаковкой — каждая пластиковая обёртка увеличивает выбросы CO₂.",
    "🚲 Замените частые поездки на машине на общественный транспорт или велосипед, когда идёте за покупками.",
    "♻️ Покупайте продукты местного производства — меньше логистика = меньше CO₂.",
    "💡 Не берите лишние пакеты на кассе — используйте многоразовые сумки.",
    "🥩 Сократите потребление красного мяса — животноводство даёт огромный углеродный след.",
    "📦 Выбирайте товары в упаковке из переработанных материалов.",
    "🌡️ Храните продукты правильно, чтобы они дольше не портились и не создавали пищевые отходы.",
    "🍎 Покупайте ровно столько, сколько съедите — пищевые отходы производят метан, который вреднее CO₂."]

lowtips = ["🎉 Отличный выбор! Вы уже следуете экологичным привычкам — так держать!",
    "🌱 Ваш чек — пример ответственного потребления. Спасибо!",
    "💚 Вы делаете вклад в чистое будущее планеты. Продолжайте в том же духе!",
    "🌟 Отличная работа! Чем больше таких чеков, тем здоровее наша планета.",
    "🌍 Вы — эко-герой! Низкий углеродный след ваших покупок вдохновляет.",
    "🌸 Спасибо, что выбираете продукты с умом. Природа ценит вашу заботу!",
    "💪 Вы доказываете, что заботиться о климате — легко и приятно!",
    "🌿 Ваш чек — пример для подражания. Так держать!",
    "🍀 Экологичный подход к покупкам — это ваш суперсил!",
    "🏆 Браво! Ваши привычки помогают планете дышать легче."]

def eco_tip():
    return random.choice(hightips)

def low_tip():
    return random.choice(lowtips)

In [ ]:
async def handle_foto(update: Update, context: ContextTypes.DEFAULT_TYPE):
    msg_stat = await update.message.reply_text("🔍 Анализирую чек...")
    try:
        photo = update.message.photo[-1]
        receipt_txt = await img_txt(photo)

        print(f"\n{'='*60}")
        print("Распознанный текст:")   # после гпт
        print('='*60)
        print(receipt_txt[:1500] if len(receipt_txt) > 1500 else receipt_txt)
        print('='*60)

        if not receipt_txt or len(receipt_txt) < 20:
            await msg_stat.edit_text("😔 Не удалось распознать текст. Пожалуйста, попробуйте:\n"
                "• сфотографировать крупнее,\n"
                "• улучшить освещение.")
            return

        found = find_prods(receipt_txt)

        if len(found) == 0:
            await msg_stat.edit_text("🔍 Продукты из чека не найдены в базе данных.")
            return

        total_co2 = sum(p['co2'] for p in found)

        response = "🌱 **ЭКОАНАЛИЗ** 🌱\n\n"
        response += f"🔍 **Найдено продуктов:** {len(found)}\n\n"
        response += "📋 **Найденные продукты:**\n"
        for f in found:
            response += f"• {f['receipt_name']}\n"
            response += f"  → {f['category']}: {f['co2']:.2f} кг CO₂/кг\n\n"

        if total_co2 > 0:
            response += f"\n📌 **Для справки**\n"
            response += f"🚗 Столько CO₂ выделяет автомобиль за {total_co2 * 4:.0f} км пути.\n"
            response += f"🌳 Столько CO₂ поглощает дерево за {total_co2 / 20:.1f} мес."

        high_co2 = any(p['co2'] > 2.0 for p in found)
        if high_co2:
            response += f"\n\n💡 **Экосовет дня**\n{eco_tip()}"
        else:
            response += f"\n\n🏆 **Экодостижение!**\n{low_tip()}"

        await msg_stat.edit_text(response, parse_mode='Markdown')

        global checks
        checks = checks + 1 # счётчик чеков зщ сессию
        print(f"Обработано чеков: {checks}")

    except:
        print("Ошибка")
        await msg_stat.edit_text("😔 Ошибка")

In [ ]:
# состояния пользователя
user_states = {}

import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.image import MIMEImage
from telegram import InlineKeyboardButton, InlineKeyboardMarkup

# настройки почты (вышкинская) - для фидбека

SMTP_server = "smtp.yandex.ru"
SMTP_port= 587
email_sender = "kavakhonina@edu.hse.ru"
email_pass = userdata.get('EMAIL_PASSWORD') # тоже в секректах
email_reciever = "kavakhonina@edu.hse.ru"

async def n_greet(update: Update, context: ContextTypes.DEFAULT_TYPE):
    # приветствие для  нового юзера
    user = update.effective_user
    keyboard = [[InlineKeyboardButton("🚀 Начать", data="start")]]
    reply = InlineKeyboardMarkup(keyboard)

    await update.message.reply_text(f"🌱 **Добро пожаловать, {user.first_name}!**\n\n"
        f"Я **EcoScan Bot** — ваш помощник в оценке углеродного следа покупок.\n\n"
        f"📸 Отправьте фото чека, и я:\n"
        f"• распознаю все продукты,\n"
        f"• рассчитаю общий CO₂ след,\n"
        f"• дам экосовет для снижения выбросов.\n\n"
        f"👇 **Нажмите «Начать», чтобы продолжить.**",
        rpl=reply, parse_mode='Markdown')

async def main_menu(message, first_name):
    """Отправляет главное меню"""
    keyboard = [[InlineKeyboardButton("📸 Анализ чека", data="scan_receipt")],
        [InlineKeyboardButton("ℹ️ О проекте", data="about")],
        [InlineKeyboardButton("📧 Связь с разработчиками", data="contact")],
        [InlineKeyboardButton("❓ Помощь", data="help")]]
    reply = InlineKeyboardMarkup(keyboard)

    await message.reply_text(f"🌱 **Привет, {first_name}!**\n\n"
        f"Я помогу вам оценить углеродный след ваших покупок.\n\n"
        f"📸 Отправьте фото чека — я найду продукты и рассчитаю их CO₂ след.\n\n"
        f"🔽 **Что хотите сделать?**",
        rpl=reply, parse_mode='Markdown')

async def button_cb(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Обработчик всех кнопок"""
    query = update.callback_query
    await query.answer()

    if query.data == "start":
        await main_menu(query.message, update.effective_user.first_name)

    elif query.data == "scan_receipt":
        keyboard = [[InlineKeyboardButton("🔙 Вернуться в меню", data="to_menu")]]
        reply = InlineKeyboardMarkup(keyboard)
        await query.edit_message_text("📸 **Отправьте фото чека**\n\n"
            "Советы для лучшего распознавания\n"
            "• Фотографируйте при хорошем освещении\n"
            "• Держите камеру ровно\n"
            "• Чек должен занимать почти весь кадр\n\n"
            "После отправки фото бот проведёт анализ и покажет результат.",
            rpl=reply, parse_mode='Markdown')

    elif query.data == "about":
        keyboard = [[InlineKeyboardButton("🔙 Главное меню", data="to_menu")]]
        reply = InlineKeyboardMarkup(keyboard)
        await query.edit_message_text("🌱 **О проекте EcoScan**\n\n"
            "Этот бот помогает оценить углеродный след продуктов из ваших чеков.\n\n"
            "📊 **Как это работает**\n"
            "• Бот распознаёт товары из чека с помощью OCR и ChatGPT\n"
            "• Сравнивает с базой данных углеродных следов продуктов\n"
            "• Показывает количество выбросов CO₂ из расчёта на 1 кг продукта\n"
            "• Даёт экосоветы для снижения воздействия на климат\n\n"
            "🌍 **Почему это важно**\n"
            "Производство продуктов питания отвечает за ~25 % мировых выбросов\n"
            "парниковых газов. Каждый осознанный выбор имеет значение!\n\n"
            "💚 **Наша миссия**\n"
            "Мы стараемся помочь каждому сделать экологичный выбор и снизить свой углеродный след.",
            rpl=reply, parse_mode='Markdown')

    elif query.data == "contact":
        keyboard = [[InlineKeyboardButton("📝 Написать разработчикам", data="fb_start")],
                    [InlineKeyboardButton("🔙 Главное меню", data="to_menu")]]
        reply = InlineKeyboardMarkup(keyboard)
        await query.edit_message_text("📧 **Связь с разработчиками**\n\n"
            "Вы можете отправить нам сообщение прямо в боте!\n\n"
            "💬 **Что можно отправить:**\n"
            "• предложения по улучшению бота,\n"
            "• сообщения об ошибках,\n"
            "• скриншоты проблемных чеков,\n"
            "• общие вопросы.\n\n"
            "👇 Нажмите «Написать разработчикам», чтобы начать.",
            rpl=reply, parse_mode='Markdown')

    elif query.data == "fb_start":
        user_states[update.effective_user.id] = {"waiting_fb": True}
        keyboard = [[InlineKeyboardButton("🔙 Отмена", data="contact")]]
        reply = InlineKeyboardMarkup(keyboard)
        await query.edit_message_text("📝 **Написать разработчикам**\n\n"
            "Напишите ваше обращение в ответ на это сообщение.\n\n"
            "После отправки ваше сообщение будет доставлено разработчикам.\n\n"
            "✏️ **Напишите ваше сообщение**",
            rpl=reply, parse_mode='Markdown')

    elif query.data == "help":
        keyboard = [[InlineKeyboardButton("🔙 Главное меню", data="to_menu")]]
        reply = InlineKeyboardMarkup(keyboard)
        await query.edit_message_text("❓ **Помощь**\n\n"
            "📸 **Как пользоваться**\n"
            "1. Нажмите «Анализ чека» в главном меню\n"
            "2. Отправьте фото чека\n"
            "3. Бот распознает товары и рассчитает углеродный след\n"
            "4. Получите экосовет\n\n"
            "🏪 **Поддерживаемые магазины**\n"
            "• Перекрёсток\n"
            "• Пятёрочка\n"
            "• Магнит\n"
            "• Лента\n"
            "• О'КЕЙ\n"
            "⚠️ **Возможные проблемы**\n"
            "• Плохое качество фото → распознавание может быть неточным\n"
            "• Не все продукты есть в базе данных (она пополняется)\n\n"
            "📊 **Оценка углеродного следа**\n"
            "• 1 кг CO₂ ≈ 4 км на автомобиле\n"
            "• 1 дерево поглощает ~20 кг CO₂ в год\n\n"
            "💡 **Экосовет дня**\n"
            "Даётся, если в чеке есть продукты с высоким CO₂ (>2 кг/кг).",
            rpl=reply, parse_mode='Markdown')

    elif query.data == "to_menu":
        await main_menu(query.message, update.effective_user.first_name)

async def feedback(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Обработка сообщений обратной связи"""
    user_id = update.effective_user.id
    user_name = update.effective_user.first_name
    username = update.effective_user.username or "нет username"

    if user_states.get(user_id, {}).get("waiting_fb"):
        user_txt = update.message.text
        user_ph = update.message.photo

        msg = MIMEMultipart()
        msg["From"] = email_sender
        msg["To"] = email_reciever
        msg["Subject"] = f"Обратная связь EcoScan от {user_name}"

        body = f"""
        <html>
        <body>
        <h2>Новое сообщение от пользователя!</h2>
        <p><b>Пользователь:</b> {user_name} (@{username})</p>
        <p><b>ID:</b> {user_id}</p>
        <p><b>Сообщение:</b></p>
        <p>{user_txt}</p>
        <p><b>Время:</b> {update.message.date}</p>
        </body>
        </html>
        """

        msg.attach(MIMEText(body, "html"))

        try:
            with smtplib.SMTP(SMTP_server, SMTP_port) as server:
                server.starttls()
                server.login(email_sender, email_pass)
                server.send_message(msg)

            if user_ph:
                photo = user_ph[-1]
                file = await photo.get_file()
                ph_bytes = await file.download_as_bytearray()

                msg_photo = MIMEMultipart()
                msg_photo["From"] = email_sender
                msg_photo["To"] = email_reciever
                msg_photo["Subject"] = f"Скриншот от {user_name}"

                image = MIMEImage(ph_bytes, name="screenshot.jpg")
                msg_photo.attach(image)

                with smtplib.SMTP(SMTP_server, SMTP_port) as server:
                    server.starttls()
                    server.login(email_sender, email_pass)
                    server.send_message(msg_photo)

            await update.message.reply_text("✅ **Сообщение отправлено!**\n\n"
                "Спасибо за вашу обратную связь! Разработчики рассмотрят ваше сообщение и ответят в ближайшее время.\n\n"
                "🔙 Вернуться в главное меню — нажмите /start",
                parse_mode='Markdown')

        except:
            print("Ошибка")
            await update.message.reply_text("❌ **Ошибка отправки**\n\n"
                "Не удалось отправить сообщение. Пожалуйста, попробуйте позже.\n\n"
                "🔙 Вернуться в главное меню — нажмите /start",
                parse_mode='Markdown')

        user_states.pop(user_id, None)
    else:
        await n_greet(update, context)

async def handle_msg(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Если пользователь написал что-то не в режиме обратной связи"""
    user_id = update.effective_user.id
    if not user_states.get(user_id, {}).get("waiting_fb"):
        await n_greet(update, context)

async def photo_menu(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Обработка фото с последующим меню"""
    await handle_foto(update, context)

    keyboard = [[InlineKeyboardButton("📸 Анализировать ещё чек", data="scan_receipt")],
        [InlineKeyboardButton("🏠 Главное меню", data="to_menu")]]
    reply = InlineKeyboardMarkup(keyboard)

    await update.message.reply_text("✨ **Что дальше?** ✨\n\n"
        "Хотите проанализировать ещё один чек или вернуться в главное меню?",
        rpl=reply, parse_mode='Markdown')

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    """Обработчик команды /start"""
    await n_greet(update, context)

print("Кнопки, меню и обратная связь загружены")

In [ ]:
# FastAPI для мониторинга

import threading
import asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from typing import Optional, List

fastapi = FastAPI(title="EcoScan Bot API", descr="API для анализа углеродного следа")

@fastapi.get("/")
def root():
    return {"status": "ok", "bot": "EcoScan Bot", "ver": "1.0.0", "descr": "Бот для анализа углеродного следа продуктов из чеков"}

@fastapi.get("/health")
def health():
    return {"status": "healthy", "database": "connected" if store_prods else "error"}

@fastapi.get("/products")
def get_prods(limit: int = 10, offset: int = 0):
    products_list = list(prod_info.values())
    return {"total": len(products_list), "products": products_list[offset:offset + limit]}

@fastapi.get("/products/{product_id}")
def get(product_id: int):
    if product_id not in prod_info:
        raise HTTPException(status_code=404, detail="Product not found")
    return prod_info[product_id]

# Запуск

def run_api():

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    config = uvicorn.Config(fastapi, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

# Отдельный поток

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

import time
time.sleep(2)

print("✅ FastAPI запущен")
print("📡 Доступные эндпойнты")
print("   GET  /")
print("   GET  /health")
print("   GET  /products")
print("   GET  /products/{product_id}")

In [ ]:
import requests

try:
    # Проверяем внутри коллаба
    response = requests.get("http://localhost:8000/")
    print("Работает внутри Colab")
    print("Ответ:", response.json())
except:
    print("Не отвечает")

In [ ]:
# Запуск бота
from telegram.ext import CallbackQueryHandler
print(f"Загружено названий из магазинов: {len(store_prods)}")
print(f"Загружено продуктов из справочника: {len(prod_info)}")
print("📸 Отправьте фото чека или нажмите /start")

async def menu_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await main_menu(update.message, update.effective_user.first_name)

async def main():
    app = Application.builder().token(BOT_TKN).build()

    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("menu", menu_cmd))

    # Обработка фото с меню после анализа
    app.add_handler(MessageHandler(filters.PHOTO, photo_menu))

    # Обработка обр связи
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, feedback))

    app.add_handler(CallbackQueryHandler(button_cb))

    await app.run_polling(drop_pending_updates=True)

if __name__ == "__main__":
    asyncio.run(main())

https://pypi.org/project/pytesseract/

https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html

https://pillow.readthedocs.io/en/stable/reference/ImageEnhance.html

https://platform.openai.com/docs/api-reference